# 01 - Destinations master list

Loads the curated `destinations_master.py` and writes `cache/destinations_master.json`
for the rest of the pipeline.

~450 destinations across three tiers:
- Ryanair-served airports (~257)
- Non-Ryanair airports (7) - kept but flagged `no_ryanair_route`
- Non-airport gems (~186) - routed via the nearest Ryanair-served airport when possible

This is a build-time notebook - re-run only when the master list changes.

## 1. Load and validate

In [ ]:
import sys, json
from datetime import datetime, timezone
from pathlib import Path

THIS_DIR = Path('.').resolve()
if str(THIS_DIR) not in sys.path:
    sys.path.insert(0, str(THIS_DIR))

from destinations_master import all_destinations, all_airports, CATEGORIES, _validate

n = _validate()
print(f"{n} destinations validated")

## 2. Summary

In [3]:
from collections import Counter

dests    = list(all_destinations())
airports = all_airports()

# Defensive: ensure consistent shape
for d in dests:
    d.setdefault("tags", [])

by_tier    = Counter(d["tier"] for d in dests)
by_iso2    = Counter(d["iso2"] for d in dests)
by_ryanair = Counter(d["ryanair_serves"] for d in dests if d["tier"] == "airport")

print("Tiers:")
for k, v in by_tier.most_common():
    print(f"  {k}: {v}")

print(f"\nAirports — Ryanair: {by_ryanair[True]} / non-Ryanair: {by_ryanair[False]}")

print(f"\nTop 10 countries:")
for iso2, n in by_iso2.most_common(10):
    print(f"  {iso2}: {n}")

print(f"\n{len(CATEGORIES)} controlled categories")

Tiers:
  airport: 264
  gem: 186

Airports — Ryanair: 257 / non-Ryanair: 7

Top 10 countries:
  IT: 51
  FR: 43
  ES: 40
  GB: 33
  DE: 26
  GR: 24
  PT: 17
  PL: 15
  CH: 14
  NL: 14

79 controlled categories


## 3. Sanity-check airport references

In [ ]:
problems = []
for d in dests:
    if d["tier"] == "gem":
        for iata, mins, eur in d["nearest_airports"]:
            if iata not in airports:
                problems.append(f"{d['id']} -> unknown {iata}")

if problems:
    print(f"{len(problems)} problems:")
    for p in problems[:10]:
        print(f"  {p}")
    raise RuntimeError("Fix destinations_master.py")
else:
    n_gems = sum(1 for d in dests if d["tier"] == "gem")
    n_refs = sum(len(d["nearest_airports"]) for d in dests if d["tier"] == "gem")
    print(f"{n_refs} airport refs across {n_gems} gems all resolve")

## 4. Write JSON

In [ ]:
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
OUT = CACHE_DIR / "destinations_master.json"

cfg = json.loads((CACHE_DIR / "config.json").read_text(encoding="utf-8"))

payload = {
    "meta": {
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "schema_version": cfg["schema_version"],
        "n_destinations": len(dests),
        "n_airports":     sum(1 for d in dests if d["tier"] == "airport"),
        "n_gems":         sum(1 for d in dests if d["tier"] == "gem"),
        "categories":     sorted(CATEGORIES),
    },
    "airports":     airports,
    "destinations": dests,
}

OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {OUT}  ({OUT.stat().st_size/1024:.1f} KB, {len(dests)} destinations)")

## 5. Done

Next: `02_flights.ipynb` to fetch Ryanair fare calendars.